# Урок 8. Контрольная работа № 1

9 класс · I четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 7](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-07.ipynb) · [Урок 9 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-09.ipynb)

---

Разбор ошибок. Самостоятельная работа: модели, графы, массивы.

In [ ]:
#@title 🚀 Шаг 1. Регистрация и подготовка урока { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
ФИО = "" #@param {type:"string"}
Класс = "9А" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-08", name=ФИО, klass=Класс)

## Где обычно ошибаются

Четверть закончена: модели, графы, деревья, массивы. Разберём ошибки,
которые встречаются чаще всего, и напишем контрольную.

### Ошибка 1. Прочерк в таблице считают нулём

В весовой матрице прочерк означает «дороги нет», а не «дорога длиной 0».
В коде отсутствие связи обозначают нулём только по договорённости —
и тогда проверять надо `!= 0`, а не суммировать.

### Ошибка 2. Забыли, что каждое ребро в матрице записано дважды

Матрица смежности неориентированного графа симметрична. Суммарная длина
всех дорог — это **половина** суммы всех клеток. Сумма степеней вершин
тоже вдвое больше числа рёбер.

### Ошибка 3. Неверный порядок при подсчёте путей

Вершину можно обрабатывать только после того, как посчитаны все вершины,
ведущие в неё. Если порядок нарушен, часть путей потеряется.

### Ошибка 4. Максимум начинают с нуля

`максимум = 0` ломается на массиве из отрицательных чисел.
Берите начальное значение из самого массива: `максимум = массив[0]`.

### Ошибка 5. Выход за границу при работе с соседями

При обращении к `массив[i + 1]` цикл должен идти до `len(массив) - 1`.

### Ошибка 6. Создание таблицы через умножение списка

`[[0] * n] * n` создаёт n ссылок на одну строку. Всегда используйте
генератор: `[[0] * n for _ in range(n)]`.

### Ошибка 7. Перепутаны индексы строки и столбца

`таблица[i][j]` — сначала строка, потом столбец. При обходе по столбцам
внешний цикл идёт по `j`, а внутренний по `i` — не наоборот.

### Чек-лист перед сдачей

1. Проверены крайние случаи: пустой массив, один элемент, все одинаковые?
2. Максимум и минимум инициализированы из данных?
3. При работе с соседями цикл укорочен на единицу?
4. Индексы строк и столбцов не перепутаны?

## Разбираем сложное

### Пример 1. Задача ОГЭ: граф и таблица вместе

> В таблице отражена протяжённость дорог. Определите длину кратчайшего
> пути из А в Е, проходящего **через пункт В**.

Хитрость в дополнительном условии. Приём: разбить путь на две части
и сложить.

In [ ]:
пункты = ["А", "Б", "В", "Г", "Д", "Е"]
веса = [
    [0, 3, 6, 0, 0, 0],
    [3, 0, 2, 5, 0, 0],
    [6, 2, 0, 4, 3, 0],
    [0, 5, 4, 0, 0, 7],
    [0, 0, 3, 0, 0, 4],
    [0, 0, 0, 7, 4, 0],
]


def кратчайшее(веса, старт, финиш):
    n = len(веса)
    d = [float("inf")] * n
    d[старт] = 0
    for _ in range(n - 1):
        for i in range(n):
            for j in range(n):
                if веса[i][j] and d[i] + веса[i][j] < d[j]:
                    d[j] = d[i] + веса[i][j]
    return d[финиш]


через_в = кратчайшее(веса, 0, 2) + кратчайшее(веса, 2, 5)
напрямую = кратчайшее(веса, 0, 5)

print(f"А → В: {кратчайшее(веса, 0, 2)}")
print(f"В → Е: {кратчайшее(веса, 2, 5)}")
print(f"Кратчайший путь через В: {через_в}")
print(f"Кратчайший путь вообще:  {напрямую}")

Обратите внимание: путь через В оказался не короче обычного —
и это нормально. Дополнительное условие всегда либо не меняет ответ,
либо ухудшает его.

Приём «разбить путь на две части» работает, потому что кратчайший путь
через заданную точку состоит из кратчайшего пути до неё и кратчайшего
пути после. Это свойство называют оптимальностью подструктуры,
и на нём построены почти все алгоритмы на графах.

### Пример 2. Двумерный массив: обход по слоям

Задача, где легко запутаться в индексах: найти сумму элементов
по «рамке» таблицы — первая и последняя строка, первый и последний столбец.

In [ ]:
таблица = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
]

строк = len(таблица)
столбцов = len(таблица[0])

сумма = 0
for i in range(строк):
    for j in range(столбцов):
        на_границе = (i == 0 or i == строк - 1 or j == 0 or j == столбцов - 1)
        if на_границе:
            сумма += таблица[i][j]

print("Таблица:")
for строка in таблица:
    print("  ", строка)
print(f"\nСумма по рамке: {сумма}")

Вместо того чтобы писать четыре отдельных цикла для четырёх сторон
(и запутаться в углах, которые посчитались бы дважды), мы проходим
всю таблицу и проверяем одно условие. Код длиннее на одну строку,
зато ошибиться в нём негде.

## Контрольная работа

Семь задач по материалу четверти.

### Задача 1. Степени вершин

По матрице смежности верните список степеней всех вершин по порядку.

In [ ]:
def степени(матрица):
    return ...

In [ ]:
si.check("1", степени, [
    ([[0, 1, 1], [1, 0, 0], [1, 0, 0]], [2, 1, 1]),
    ([[0, 0], [0, 0]], [0, 0]),
    ([[0, 4, 7], [4, 0, 2], [7, 2, 0]], [2, 2, 2]),
])

### Задача 2. Количество путей

По ориентированному графу (словарь «вершина → список потомков»)
верните количество путей из старта в финиш. Обрабатывайте вершины
в алфавитном порядке.

In [ ]:
def путей(граф, старт, финиш):
    return ...

In [ ]:
si.check("2", путей, [
    (({"А": ["Б", "В"], "Б": ["Г"], "В": ["Г"], "Г": []}, "А", "Г"), 2),
    (({"А": ["Б"], "Б": ["В"], "В": []}, "А", "В"), 1),
    (({"А": ["Б", "В"], "Б": ["Г", "Д"], "В": ["Д"], "Г": ["Е"], "Д": ["Е"], "Е": []},
      "А", "Е"), 3),
])

### Задача 3. Высота дерева

По дереву и корню верните его высоту. У дерева из одной вершины
высота 0.

In [ ]:
def высота(дерево, корень):
    return ...

In [ ]:
si.check("3", высота, [
    (({"А": ["Б", "В"], "Б": [], "В": []}, "А"), 1),
    (({"А": []}, "А"), 0),
    (({"А": ["Б"], "Б": ["В"], "В": ["Г"], "Г": []}, "А"), 3),
])

### Задача 4. Обработка массива

Верните количество элементов, которые больше своего левого соседа.
Первый элемент не учитывается — у него нет левого соседа.

In [ ]:
def больше_левого(массив):
    return ...

In [ ]:
si.check("4", больше_левого, [
    ([1, 2, 3], 2),
    ([3, 2, 1], 0),
    ([1, 5, 2, 8], 2),
    ([7], 0),
    ([], 0),
])

### Задача 5. Двумерный массив

Верните список сумм по каждому столбцу таблицы.

In [ ]:
def суммы_столбцов(таблица):
    return ...

In [ ]:
si.check("5", суммы_столбцов, [
    ([[1, 2, 3], [4, 5, 6]], [5, 7, 9]),
    ([[1], [2], [3]], [6]),
    ([[0, 0], [0, 0]], [0, 0]),
])

### Задача 6. Кратчайший путь

По весовой матрице верните длину кратчайшего пути между двумя вершинами.
Если пути нет — верните `-1`.

In [ ]:
def кратчайший(веса, старт, финиш):
    return ...

In [ ]:
si.check("6", кратчайший, [
    (([[0, 3, 6, 0], [3, 0, 2, 5], [6, 2, 0, 4], [0, 5, 4, 0]], 0, 3), 8),
    (([[0, 1], [1, 0]], 0, 1), 1),
    (([[0, 0], [0, 0]], 0, 1), -1),
])

### Задача 7. Теоретический вопрос

В неориентированном графе 6 вершин, и сумма их степеней равна 14.
Сколько в графе рёбер?

In [ ]:
рёбер = 0

si.check_value("7", рёбер, "7902699be42c8a8e",
               hint="Каждое ребро вносит вклад в степени двух вершин.")

## Домашнее задание на каникулы

### Домашнее задание. Анализатор графа

Соберите функцию, которая по матрице смежности возвращает сводку
о графе — список из четырёх чисел:

1. количество вершин;
2. количество рёбер;
3. степень самой связной вершины;
4. количество изолированных вершин (без единой связи).

`анализ([[0,1,0],[1,0,0],[0,0,0]])` → `[3, 1, 1, 1]`

In [ ]:
def анализ(матрица):
    return ...

In [ ]:
si.check("дз", анализ, [
    ([[0, 1, 0], [1, 0, 0], [0, 0, 0]], [3, 1, 1, 1]),
    ([[0, 1, 1], [1, 0, 1], [1, 1, 0]], [3, 3, 2, 0]),
    ([[0, 0], [0, 0]], [2, 0, 0, 2]),
])

---

### Итоги четверти

Вы научились строить модели и оценивать их адекватность, работать
с графами в двух представлениях, находить кратчайшие пути, разбираться
в деревьях и уверенно обрабатывать одномерные и двумерные массивы.

Во второй четверти мы займёмся программированием всерьёз: функции
и модульность, рекурсия, словари, работа с файлами. А в конце четверти
будет первый практикум по заданиям ОГЭ — там всё это понадобится сразу.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 7](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-07.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 9 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-09.ipynb)